In [1]:
#Ishaan
#Patient split
import numpy as np
import pandas as pd

NODULES   = "data/nodules_labels.csv"
SPLIT_MAP = "data/patient_split.csv"     # patient_id -> split, the single source of truth
SEED      = 42
FRACTIONS = (0.70, 0.15, 0.15)           # train, val, test

nodules = pd.read_csv(NODULES)

# Split by PATIENT, never by nodule or series. A patient can have several nodules
# and (7 of them here) two scans; if any of their data is in train, none of it may
# be in test, or the model is scored on a body it has already seen.
patients = (nodules.groupby("patient_id")["hard"].max()      #their most suspicious nodule
                   .rename("stratum").reset_index())

# Stratify so each split gets the same mix of easy and suspicious patients.
# Without this, test ends up with ~25 class-5 nodules and the number swings with the seed.
rng = np.random.default_rng(SEED)
assign = {}
for stratum, group in patients.groupby("stratum"):
    pids = group["patient_id"].to_numpy()
    rng.shuffle(pids)
    a = round(FRACTIONS[0] * len(pids))
    b = a + round(FRACTIONS[1] * len(pids))
    for pid in pids[:a]:  assign[pid] = "train"
    for pid in pids[a:b]: assign[pid] = "val"
    for pid in pids[b:]:  assign[pid] = "test"

split_map = pd.DataFrame({"patient_id": list(assign), "split": list(assign.values())})
split_map = split_map.sort_values("patient_id")
split_map.to_csv(SPLIT_MAP, index=False)

nodules["split"] = nodules["patient_id"].map(assign)
nodules.to_csv(NODULES, index=False)


print("patients per split:\n", split_map["split"].value_counts().reindex(["train","val","test"]))
print("\nnodules per split:\n", nodules["split"].value_counts().reindex(["train","val","test"]))
print("\nshare of each hard class per split:")
print(nodules.groupby("split")["hard"].value_counts(normalize=True).unstack()
             .reindex(["train","val","test"]).round(3).to_string())
print("\nshare suspicious (hard >= 4):")
print((nodules.assign(sus=nodules.hard >= 4).groupby("split")["sus"].mean()
              .reindex(["train","val","test"]).round(3)).to_string())
print("\nshare with reader disagreement (spread >= 2):")
print((nodules.assign(d=nodules.spread >= 2).groupby("split")["d"].mean()
              .reindex(["train","val","test"]).round(3)).to_string())
print(f"\nsaved {SPLIT_MAP} and added 'split' to {NODULES}")


patients per split:
 split
train    557
val      119
test     120
Name: count, dtype: int64

nodules per split:
 split
train    1305
val       290
test      290
Name: count, dtype: int64

share of each hard class per split:
hard       1      2      3      4      5
split                                   
train  0.107  0.149  0.477  0.170  0.097
val    0.103  0.162  0.479  0.172  0.083
test   0.128  0.159  0.469  0.152  0.093

share suspicious (hard >= 4):
split
train    0.267
val      0.255
test     0.245

share with reader disagreement (spread >= 2):
split
train    0.410
val      0.410
test     0.359

saved data/patient_split.csv and added 'split' to data/nodules_labels.csv
